# ipynb/rgcnformer_plant_cls_res_nofont.ipynb - RGCNFormer Plant 分类结果(无字体) / RGCNFormer Plant classification (no font)

## 项目背景 / Background
RGCNFormer Plant 分类结果(无字体)
RGCNFormer Plant classification (no font)

## 功能模块 / Modules
- Plant 分类结果(无字体)
- (详见各代码单元 / see code cells)

## 输入 / Inputs
- 上一阶段产物(.npy/.pt/.csv/.json)/ prior-stage outputs
- 内嵌常量与参数 / inline constants and params

## 输出 / Outputs
- 图表(内联显示) / figures (inline)
- 中间变量 / intermediate variables
- 导出文件(.png/.pdf/.csv) / exported files

## 数据流 / Data Flow
1. 加载数据 / Load data
2. 运行分析 / Run analysis
3. 渲染图表 / Render figures
4. 导出 / Export

## 相关文件 / Related Files
- 调用 / Calls: train_plant.py、inference_*.py
- 被调用 / Called by: 报告 / 论文 / report / paper

## 使用示例 / Usage Example
- 在 JupyterLab 中打开 / open in JupyterLab
- 逐单元运行 / run cells sequentially

## 作者 / Author
项目组 / Project Team

## 版本 / Version
1.0



In [1]:
# 1. 安装并加载字体管理包
library(showtext)

# 加载必要的包
library(ggplot2)
library(dplyr)
library(tidyr)
library(patchwork)
library(scales)

# 注册 DejaVu Sans 字体
font_add("DejaVu", "DejaVuSans.ttf")

# 开启自动渲染
showtext_auto()

# 设置 dpi
showtext_opts(dpi = 300)

# 定义要使用的指标
metric_levels <- c("Rec", "Acc", "AUC", "Sn", "Sp")

# 定义配色方案
morandi_hues <- c(
  "Rec" = "#B28F8F",
  "Acc" = "#768D99", 
  "AUC" = "#B6A691",
  "Sn" = "#88A096", 
  "Sp" = "#A8A085"
)

# 获取当前工作目录
current_wd <- getwd()
cat("Current working directory:", current_wd, "\n")

# 设置数据目录和输出目录
# 检查当前工作目录的最后一部分是否是 ipynb
if (basename(current_wd) == "ipynb") {
  data_dir <- "plant_data"
  output_dir <- "plant_png"
} else {
  data_dir <- "ipynb/plant_data"
  output_dir <- "ipynb/plant_png"
}

cat("Data directory:", data_dir, "\n")
cat("Output directory:", output_dir, "\n")

# 获取所有CSV文件
csv_files <- list.files(data_dir, pattern = "\\.csv$", full.names = TRUE)

# 打印找到的文件
cat("Found CSV files:\n")
print(csv_files)

# 如果还是找不到，尝试多种可能的路径
if (length(csv_files) == 0) {
  cat("Trying alternative paths...\n")
  possible_paths <- c(
    "plant_data",
    "ipynb/plant_data",
    "../plant_data",
    file.path(dirname(current_wd), "plant_data")
  )
  
  for (path in possible_paths) {
    if (dir.exists(path)) {
      cat("Found directory:", path, "\n")
      csv_files <- list.files(path, pattern = "\\.csv$", full.names = TRUE)
      if (length(csv_files) > 0) {
        data_dir <- path
        output_dir <- file.path(dirname(path), "plant_png")
        cat("Using path:", path, "\n")
        break
      }
    }
  }
  cat("Found CSV files (after search):\n")
  print(csv_files)
}

# 定义Shot的固定水平顺序（从上到下：0, 1, 5, 10 shot）
shot_levels <- c("0shot", "1shot", "5shot", "10shot")

# 循环处理每个CSV文件
for (i in seq_along(csv_files)) {
  csv_file <- csv_files[i]
  
  cat("\nProcessing:", csv_file, "\n")
  
  tryCatch({
    # 读取数据
    df <- read.csv(csv_file)
    cat("Data loaded. Rows:", nrow(df), "\n")
    
    # 将Shot数除以2
    df$Shot <- df$Shot / 2
    
    # 将Shot转换为字符型，用于y轴显示
    df$Shot <- paste0(df$Shot, "shot")
    
    # 数据准备：只保留需要的指标
    plot_data <- df %>%
      select(Shot, all_of(metric_levels)) %>%
      pivot_longer(cols = -Shot, names_to = "Metric", values_to = "Value") %>%
      mutate(
        Metric = factor(Metric, levels = metric_levels),
        Shot = factor(Shot, levels = shot_levels)
      )
    
    # 为每个数据点计算颜色
    # 先按 Metric 分组，然后为每组计算颜色
    plot_data <- plot_data %>%
      group_by(Metric) %>%
      mutate(
        # 使用 !! 和 := 来正确赋值函数结果
        point_color = col_numeric(
          palette = c("#EAECEE", morandi_hues[as.character(Metric[1])]), 
          domain = c(0.35, 1)
        )(Value)
      ) %>%
      ungroup()
    
    cat("Plot data prepared. Rows:", nrow(plot_data), "\n")
    
    # 创建背景层数据
    bg_data <- data.frame(
      Metric = factor(metric_levels, levels = metric_levels),
      xmin = seq_along(metric_levels) - 0.5,
      xmax = seq_along(metric_levels) + 0.5,
      bg_fill = rep("#ffffff", length.out = length(metric_levels))
    )
    
    # 获取文件名（不含路径和扩展名）
    base_name <- tools::file_path_sans_ext(basename(csv_file))
    
    # 创建图表
    p1 <- ggplot() +
      geom_rect(data = bg_data, aes(xmin = xmin, xmax = xmax, ymin = -Inf, ymax = Inf, fill = bg_fill), 
                alpha = 1, show.legend = FALSE) +
      scale_fill_identity() + 
      
      # 绘制圆圈
      geom_point(data = plot_data, aes(x = Metric, y = Shot, size = Value, color = point_color)) +
      
      scale_color_identity() + 
      
      # 配置尺寸图例
      scale_size_continuous(
        name = "Performance Value",
        range = c(10, 22),
        limits = c(0.2, 1), 
        breaks = c(0.2,0.4, 0.6, 0.8),
        labels = c("20%","40%", "60%", "80%")
      ) +
      
      guides(
        size = guide_legend(
          override.aes = list(color = "#85929E"),
          order = 1
        )
      ) +
      
      scale_x_discrete(position = "top", expand = expansion(add = c(0.6, 0.6))) + 
      scale_y_discrete(limits = rev(shot_levels), expand = expansion(add = c(0.8, 0.8))) +
      theme_minimal() +
      theme(
        text = element_text(family = "DejaVu"),
        axis.text.x = element_text(size = 12, face = "bold", color = "#4A4A4A"),
        axis.text.y = element_text(size = 12, color = "#4A4A4A"),
        axis.title = element_blank(),
        panel.grid = element_blank(),
        legend.position = "right",
        legend.title = element_text(size = 12, face = "bold", color = "#4A4A4A"),
        legend.text = element_text(size = 12, color = "#4A4A4A")
      )
    
    # 确保输出目录存在
    if (!dir.exists(output_dir)) {
      dir.create(output_dir, recursive = TRUE)
    }
    
    # 保存为PDF
    output_path <- file.path(output_dir, paste0(base_name, ".pdf"))
    ggsave(output_path, p1, width = 10, height = 4, dpi = 300)
    
    # 打印处理进度
    cat("Successfully saved:", output_path, "\n")
  }, error = function(e) {
    cat("Error processing", csv_file, ":", conditionMessage(e), "\n")
  })
}

cat("\nAll files processed!\n")

Loading required package: sysfonts

Loading required package: showtextdb


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




Current working directory: /home/dc/vscode/vscode20260124/rgcnformer_sum/ipynb 
Data directory: plant_data 
Output directory: plant_png 
Found CSV files:
[1] "plant_data/m5c.csv" "plant_data/m6a.csv" "plant_data/y.csv"  



Processing: plant_data/m5c.csv 
Data loaded. Rows: 4 
Plot data prepared. Rows: 20 
Successfully saved: plant_png/m5c.pdf 

Processing: plant_data/m6a.csv 
Data loaded. Rows: 4 
Plot data prepared. Rows: 20 
Successfully saved: plant_png/m6a.pdf 

Processing: plant_data/y.csv 
Data loaded. Rows: 4 


Warning message:
“There were 2 warnings in `mutate()`.
The first warning was:
ℹ In argument: `point_color = col_numeric(palette = c("#EAECEE",
  morandi_hues[as.character(Metric[1])]), ...`.
ℹ In group 1: `Metric = Rec`.
Caused by warning:
! Some values were outside the color scale and will be treated as NA
ℹ Run `dplyr::last_dplyr_warnings()` to see the 1 remaining warning.”


Plot data prepared. Rows: 20 
Successfully saved: plant_png/y.pdf 

All files processed!
